In [21]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
from datetime import datetime
import re
from tqdm import tqdm

def get_match_details(link):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'}
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.content, 'html.parser')
    date = ""
    attendance = ""
    for elem in soup.find_all(name='div', attrs={"class":"mc-summary__info"}):
        try:
            date = datetime.strptime(elem.text.strip(), "%a %d %b %Y")
        except ValueError:
            pass
        try:
            if "Att" in elem.text:
                attendance = int(elem.text.strip().replace("Att:", "").replace(',', ''))
        except ValueError:
            pass

    home_team, away_team = [x.find_all(name="span", attrs={'class':'u-hide-phablet'})[0].text for x in soup.find_all(name='div', attrs={"class":"mc-summary__team"})]
    home_team_goals, away_team_goals = map(int, soup.find_all(name='div', attrs={"class":"mc-summary__score"})[0].text.split(' - '))
    home_team_goals_half_time, away_team_goals_half_time = map(int, soup.find_all(name='div', attrs={"class":"mc-summary__half-time"})[0].findChild("span").text.split('-'))
    
    event_list = []
    for elem in soup.find_all(name='div', attrs={"class":"timeLineEventsContainer"})[0].findChildren(name="div", attrs={'class' : 'event'}):
        for list_elements in elem.findChildren("ul", attrs={'class' : 'event__icons'}):
            team = [cls for cls in list_elements.get_attribute_list("class") if "--" in cls][0].split('--')[-1]
            for events in list_elements.findChildren("li", attrs={'class' : 'event__icon'}):
                if "event__icon--dummy" in events.get_attribute_list("class"):
                    continue
                try:
                    header = events.findChildren("div")[0]
                    header = header.findChildren("div")[0]
                    header = header.findChildren("header")[0]
                    time = header.findChildren("time")[0].text
                    event = header.findChildren("span")[0].text
                    event_list.append((team, time, event))
                except IndexError:
                    pass
    return {
        'match_date': date,
        'attendance': attendance,
        'home_team': home_team,
        'away_team': away_team,
        'home_team_goals': home_team_goals,
        'away_team_goals': away_team_goals,
        'home_team_goals_half_time': home_team_goals_half_time,
        'away_team_goals_half_time': away_team_goals_half_time,
        'events': event_list
    }

def minute_parser(time_string):
    time_string = time_string.replace("'", '').replace('"', '')
    if "90 +" in time_string or "45 +" in time_string:
        return time_string  # Returning the time string unchanged if it contains "90 +" or "45 +"
    return int(time_string)  # Convert to integer otherwise

# Main loop to gather match data
matches = list(range(115947, 115949))
match_data = []
for match in tqdm(matches):
    link = f"https://www.premierleague.com/match/{match}"
    try:
        dic = get_match_details(link)
        dic['link'] = link
    except Exception as e:
        print("Exception:", str(e), "\n", link)
        dic = {
            'match_date': "",
            'attendance': 0,
            'home_team': "",
            'away_team': "",
            'home_team_goals': "",
            'away_team_goals': "",
            'home_team_goals_half_time': "",
            'away_team_goals_half_time': "",
            'events': [],
            "link": link
        }
    match_data.append(dic)

df = pd.DataFrame(match_data)
df['events'] = df['events'].apply(lambda x: [(i[0], minute_parser(i[1]), i[2]) for i in x])

df.to_csv('demo_data.csv', index=False)


100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.30it/s]
